# CUB-200 Data Exploration

This notebook is used to inspect and validate the CUB-200 dataset before model training.

The main purposes are:

- Check dataset paths and annotation files
- Check the number of training and testing images
- Check the number of classes
- Check class distribution
- Display random bird images
- Inspect image dimensions
- Check for missing images
- Check for corrupted images
- Test PyTorch Dataset and DataLoader
- Visualize basic data augmentation

This notebook is for **data exploration and validation only**.
The actual model training will be performed using `src/train.py`.

In [ ]:
# Import required libraries

from pathlib import Path
from collections import Counter
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print( "PyTorch version:", torch.__version__ )
print( "CUDA available:", torch.cuda.is_available() )

if torch.cuda.is_available():
	print( "GPU:", torch.cuda.get_device_name( 0 ) )

## 1. Project Paths

The notebook is located inside the `notebooks` folder.

Therefore, the project root directory is one level above the current notebook directory.

Expected structure:

```text
COS30082_Assignment1/
├── data/
│   ├── Train/
│   ├── Test/
│   ├── train.txt
│   └── test.txt
│
├── src/
│   ├── dataset.py
│   ├── models.py
│   ├── train.py
│   ├── evaluate.py
│   └── utils.py
│
└── notebooks/
    └── data_exploration.ipynb
```

In [ ]:
# Define project and dataset paths

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR = DATA_DIR / "Test"

TRAIN_TXT = DATA_DIR / "train.txt"
TEST_TXT = DATA_DIR / "test.txt"

print( "Project root :", PROJECT_ROOT )
print( "Data folder  :", DATA_DIR )

print( "Train folder :", TRAIN_DIR )
print( "Test folder  :", TEST_DIR )

print( "train.txt    :", TRAIN_TXT )
print( "test.txt     :", TEST_TXT )

In [ ]:
# Check whether the expected files and directories exist

paths_to_check = {
	"Train directory": TRAIN_DIR, "Test directory": TEST_DIR, "train.txt": TRAIN_TXT, "test.txt": TEST_TXT,
}

for name, path in paths_to_check.items():

	status = "FOUND" if path.exists() else "NOT FOUND"

	print( f"{name:20s}: {status}" )
	print( f"    {path}" )

## 2. Read Annotation Files

The assignment provides annotations in the following format:

```text
image_name.jpg class_label
```

For example:

```text
Black_footed_Albatross_0004_2731401028.jpg 0
```

The image name identifies the image while the class label identifies the bird species.

In [ ]:
def read_annotations(annotation_file):
	"""
    Read an annotation file.

    Each line should contain:
        image_name label

    Returns:
        pandas.DataFrame
    """

	records = []

	with open( annotation_file, "r", encoding = "utf-8" ) as f:

		for line_number, line in enumerate( f, start = 1 ):

			line = line.strip()

			# Ignore empty lines
			if not line:
				continue

			parts = line.split()

			# Each valid line should contain exactly two values
			if len( parts ) != 2:
				print(
					f"Warning: invalid line {line_number}: {line}"
				)
				continue

			image_name, label = parts

			records.append(
				(
					image_name, int( label )
				)
			)

	return pd.DataFrame(
		records, columns = ["image_name", "label"]
	)


# Read training and testing annotations

train_df = read_annotations( TRAIN_TXT )
test_df = read_annotations( TEST_TXT )

print( "Training samples:", len( train_df ) )
print( "Testing samples :", len( test_df ) )

In [ ]:
# Display the first few annotation records

print( "Training annotations:" )
display( train_df.head( 10 ) )

print( "\nTesting annotations:" )
display( test_df.head( 10 ) )

## 3. Check Dataset Classes

CUB-200 contains 200 bird species.

The following checks verify:

- Number of unique classes
- Minimum class label
- Maximum class label
- Missing class labels
- Duplicate annotation rows

In [ ]:
# Basic class information

print( "Training classes:" )
print( "Number of classes:", train_df["label"].nunique() )
print( "Minimum label:", train_df["label"].min() )
print( "Maximum label:", train_df["label"].max() )

print( "\nTesting classes:" )
print( "Number of classes:", test_df["label"].nunique() )
print( "Minimum label:", test_df["label"].min() )
print( "Maximum label:", test_df["label"].max() )

In [ ]:
# Check whether any class labels are missing

expected_labels = set(
	range(
		train_df["label"].min(), train_df["label"].max() + 1
	)
)

actual_labels = set( train_df["label"] )

missing_labels = sorted(
	expected_labels - actual_labels
)

if missing_labels:
	print( "Missing training labels:" )
	print( missing_labels )
else:
	print( "No missing training labels." )

In [ ]:
# Check duplicate annotation records

duplicate_train = train_df.duplicated().sum()
duplicate_test = test_df.duplicated().sum()

print( "Duplicate training records:", duplicate_train )
print( "Duplicate testing records :", duplicate_test )

## 4. Check Image Paths

The annotation files contain image names.

This section checks whether every image referenced by the annotation files actually exists inside the corresponding dataset folder.

In [ ]:
def find_image(base_dir, image_name):
	"""
    Find an image recursively inside base_dir.

    First checks the direct path.
    If it does not exist, searches subdirectories.
    """

	direct_path = base_dir / image_name

	if direct_path.exists():
		return direct_path

	matches = list(
		base_dir.rglob( image_name )
	)

	if matches:
		return matches[0]

	return None

In [ ]:
def check_image_paths(df, image_dir):
	"""
    Check whether all images referenced by a DataFrame exist.
    """

	missing = []

	for image_name in df["image_name"]:

		image_path = find_image(
			image_dir, image_name
		)

		if image_path is None:
			missing.append( image_name )

	return missing


missing_train = check_image_paths(
	train_df, TRAIN_DIR
)

missing_test = check_image_paths(
	test_df, TEST_DIR
)

print(
	"Missing training images:", len( missing_train )
)

print(
	"Missing testing images:", len( missing_test )
)

In [ ]:
# Display missing images if any exist

if missing_train:

	print( "Missing training images:" )

	for image in missing_train[:20]:
		print( image )

else:

	print( "All training images were found." )

print()

if missing_test:

	print( "Missing testing images:" )

	for image in missing_test[:20]:
		print( image )

else:

	print( "All testing images were found." )

## 5. Class Distribution

Class distribution is important because imbalanced classes can affect model performance.

We will calculate:

- Number of training images per class
- Number of testing images per class
- Minimum number of images
- Maximum number of images
- Average number of images

In [ ]:
# Calculate the number of images for each class

train_counts = (train_df["label"].value_counts().sort_index())

test_counts = (test_df["label"].value_counts().sort_index())

distribution_df = pd.DataFrame(
	{
		"train_count": train_counts, "test_count": test_counts
	}
).fillna( 0 ).astype( int )

display(
	distribution_df.head( 20 )
)

In [ ]:
# Training distribution statistics

print( "Training images per class" )

print(
	"Minimum:", train_counts.min()
)

print(
	"Maximum:", train_counts.max()
)

print(
	"Mean:", round( train_counts.mean(), 2 )
)

print( "\nTesting images per class" )

print(
	"Minimum:", test_counts.min()
)

print(
	"Maximum:", test_counts.max()
)

print(
	"Mean:", round( test_counts.mean(), 2 )
)

In [ ]:
# Plot training class distribution

plt.figure( figsize = (16, 5) )

plt.bar(
	train_counts.index, train_counts.values
)

plt.xlabel( "Class label" )
plt.ylabel( "Number of training images" )

plt.title(
	"CUB-200 Training Class Distribution"
)

plt.tight_layout()

plt.show()

## 6. Display Random Training Images

Visual inspection is useful for confirming that:

- Images can be loaded correctly
- Images contain birds
- Annotation labels are associated with the expected images

In [ ]:
def get_image_path(image_dir, image_name):
	"""
    Return the full path of an image.
    """

	image_path = find_image(
		image_dir, image_name
	)

	if image_path is None:

		raise FileNotFoundError(
			f"Image not found: {image_name}"
		)

	return image_path

In [ ]:
def show_random_images(
	df, image_dir, n = 12
):
	"""
    Display random images from the dataset.
    """

	samples = df.sample(
		min( n, len( df ) ), random_state = 42
	)

	columns = 4

	rows = int(
		np.ceil(
			len( samples ) / columns
		)
	)

	plt.figure(
		figsize = (14, 3.5 * rows)
	)

	for i, (_, row) in enumerate(
		samples.iterrows()
	):

		image_path = get_image_path(
			image_dir, row["image_name"]
		)

		image = Image.open(
			image_path
		).convert( "RGB" )

		plt.subplot(
			rows, columns, i + 1
		)

		plt.imshow( image )

		plt.title(
			f"Label: {row['label']}\n"
			f"{row['image_name']}", fontsize = 8
		)

		plt.axis( "off" )

	plt.tight_layout()

	plt.show()


show_random_images(
	train_df, TRAIN_DIR, n = 12
)

## 7. Inspect Image Sizes

The original CUB-200 images do not necessarily have the same resolution.

Before being passed into a CNN such as ResNet18, the images will be resized and cropped to a fixed size, such as:

```text
224 × 224
```

This section checks the original image dimensions.

In [ ]:
def inspect_image_sizes(
	df, image_dir
):
	"""
    Inspect the width and height
    of every image.
    """

	sizes = []
	failed = []

	for image_name in df["image_name"]:

		try:

			image_path = get_image_path(
				image_dir, image_name
			)

			with Image.open( image_path ) as image:

				sizes.append(
					(
						image.width, image.height
					)
				)

		except Exception as e:

			failed.append(
				(
					image_name, str( e )
				)
			)

	return sizes, failed

In [ ]:
# Inspect training image dimensions

image_sizes, size_errors = inspect_image_sizes(
	train_df, TRAIN_DIR
)

widths = [size[0] for size in image_sizes]

heights = [size[1] for size in image_sizes]

print(
	"Images inspected:", len( image_sizes )
)

print(
	"Failed images:", len( size_errors )
)

print( "\nWidth statistics" )

print(
	"Minimum:", min( widths )
)

print(
	"Maximum:", max( widths )
)

print(
	"Mean:", round( np.mean( widths ), 2 )
)

print( "\nHeight statistics" )

print(
	"Minimum:", min( heights )
)

print(
	"Maximum:", max( heights )
)

print(
	"Mean:", round( np.mean( heights ), 2 )
)

In [ ]:
# Display the most common image dimensions

size_counter = Counter(
	image_sizes
)

size_table = pd.DataFrame(
	[(
		width, height, count
	)

		for (
		width, height
	), count in size_counter.most_common( 15 )],

	columns = [
		"width", "height", "count"
	]
)

display( size_table )

## 8. Check for Corrupted Images

Every image is opened and verified using PIL.

If an image cannot be opened or verified, it will be reported as corrupted.

In [ ]:
def check_corrupted_images(
	df, image_dir
):
	"""
    Check whether images can be
    successfully opened and verified.
    """

	corrupted = []

	for image_name in df["image_name"]:

		try:

			image_path = get_image_path(
				image_dir, image_name
			)

			with Image.open( image_path ) as image:

				image.verify()

		except Exception as e:

			corrupted.append(
				(
					image_name, str( e )
				)
			)

	return corrupted

In [ ]:
# Check training and testing images

corrupted_train = check_corrupted_images(
	train_df, TRAIN_DIR
)

corrupted_test = check_corrupted_images(
	test_df, TEST_DIR
)

print(
	"Corrupted training images:", len( corrupted_train )
)

print(
	"Corrupted testing images:", len( corrupted_test )
)

In [ ]:
# Display corrupted image information if any exists

if corrupted_train:

	print( "Training image errors:" )

	for item in corrupted_train[:20]:
		print( item )

else:

	print( "No corrupted training images found." )

print()

if corrupted_test:

	print( "Testing image errors:" )

	for item in corrupted_test[:20]:
		print( item )

else:

	print( "No corrupted testing images found." )

## 9. Test a PyTorch Dataset

The actual training pipeline will use a PyTorch `Dataset`.

Here we create a small exploration version to confirm that:

1. An image can be loaded
2. The image can be converted to a tensor
3. The label is correctly returned
4. The tensor has the expected shape

In [ ]:
class CUBExplorationDataset( Dataset ):

	def __init__(
		self, dataframe, image_dir, transform = None
	):

		self.dataframe = (dataframe.reset_index( drop = True ))

		self.image_dir = image_dir

		self.transform = transform

	def __len__(self):

		return len(
			self.dataframe
		)

	def __getitem__(
		self, index
	):

		row = self.dataframe.iloc[index]

		image_path = get_image_path(
			self.image_dir, row["image_name"]
		)

		image = Image.open(
			image_path
		).convert( "RGB" )

		label = int(
			row["label"]
		)

		if self.transform:

			image = self.transform(
				image
			)

		return image, label

In [ ]:
# Basic transformation for testing the Dataset

exploration_transform = transforms.Compose(
	[

		transforms.Resize(
			(224, 224)
		),

		transforms.ToTensor()

	]
)

train_dataset = CUBExplorationDataset(

	train_df,

	TRAIN_DIR,

	transform = exploration_transform
)

print(
	"Dataset length:", len( train_dataset )
)

In [ ]:
# Inspect one Dataset item

image_tensor, label = train_dataset[0]

print(
	"Image tensor shape:", image_tensor.shape
)

print(
	"Image tensor dtype:", image_tensor.dtype
)

print(
	"Image minimum value:", image_tensor.min().item()
)

print(
	"Image maximum value:", image_tensor.max().item()
)

print(
	"Label:", label
)

## 10. Test the DataLoader

The DataLoader provides batches of images to the training loop.

For a batch size of 16 and an input resolution of 224 × 224, the expected image tensor shape is:

```text
[16, 3, 224, 224]
```

where:

- 16 = batch size
- 3 = RGB channels
- 224 = image height
- 224 = image width

In [ ]:
# Create a DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(

	train_dataset,

	batch_size = BATCH_SIZE,

	shuffle = True,

	num_workers = 0,

	pin_memory = torch.cuda.is_available()
)

# Get one batch

images, labels = next(
	iter( train_loader )
)

print(
	"Batch image shape:", images.shape
)

print(
	"Batch label shape:", labels.shape
)

print(
	"Labels:", labels.tolist()
)

In [ ]:
# Visualize images from a DataLoader batch

def show_tensor_batch(
	images, labels, n = 8
):

	n = min(
		n, len( images )
	)

	columns = 4

	rows = int(
		np.ceil(
			n / columns
		)
	)

	plt.figure(
		figsize = (14, 3.5 * rows)
	)

	for i in range( n ):

		image = (images[i].permute( 1, 2, 0 ).numpy())

		image = np.clip(
			image, 0, 1
		)

		plt.subplot(
			rows, columns, i + 1
		)

		plt.imshow( image )

		plt.title(
			f"Label: {labels[i].item()}"
		)

		plt.axis( "off" )

	plt.tight_layout()

	plt.show()


show_tensor_batch(
	images, labels
)

## 11. Preview Data Augmentation

Data augmentation can help reduce overfitting.

For the training dataset, we can apply transformations such as:

- Random resized crop
- Random horizontal flip
- Color jitter

These transformations create different versions of the same training image.

The actual final augmentation configuration will be defined in the training pipeline.

In [ ]:
# Example training augmentation pipeline

augmentation_transform = transforms.Compose(
	[

		transforms.RandomResizedCrop(
			224, scale = (0.8, 1.0)
		),

		transforms.RandomHorizontalFlip(
			p = 0.5
		),

		transforms.ColorJitter(
			brightness = 0.2, contrast = 0.2, saturation = 0.2, hue = 0.05
		),

		transforms.ToTensor()
	]
)

In [ ]:
# Select one random training image

sample_row = train_df.sample(
	1, random_state = 123
).iloc[0]

sample_path = get_image_path(
	TRAIN_DIR, sample_row["image_name"]
)

sample_image = Image.open(
	sample_path
).convert( "RGB" )

print(
	"Image:", sample_row["image_name"]
)

print(
	"Class label:", sample_row["label"]
)

In [ ]:
# Display original image and several augmented versions

plt.figure(
	figsize = (14, 7)
)

# Original image

plt.subplot(
	2, 4, 1
)

plt.imshow(
	sample_image
)

plt.title( "Original" )

plt.axis( "off" )

# Augmented images

for i in range( 2, 9 ):

	augmented = augmentation_transform(
		sample_image
	)

	augmented = (augmented.permute( 1, 2, 0 ).numpy())

	augmented = np.clip(
		augmented, 0, 1
	)

	plt.subplot(
		2, 4, i
	)

	plt.imshow(
		augmented
	)

	plt.title(
		f"Augmented {i - 1}"
	)

	plt.axis( "off" )

plt.tight_layout()

plt.show()

## 12. Final Dataset Summary

This section provides a final sanity check before model training.

The dataset should ideally satisfy:

- 4829 training images
- 200 classes
- All training images found
- All testing images found
- No corrupted images
- DataLoader produces tensors with the expected shape

In [ ]:
print( "=" * 60 )

print( "CUB-200 DATASET SUMMARY" )

print( "=" * 60 )

print(
	f"Training images        : {len( train_df )}"
)

print(
	f"Testing images         : {len( test_df )}"
)

print(
	f"Training classes       : {train_df['label'].nunique()}"
)

print(
	f"Testing classes        : {test_df['label'].nunique()}"
)

print(
	f"Missing train images   : {len( missing_train )}"
)

print(
	f"Missing test images    : {len( missing_test )}"
)

print(
	f"Corrupted train images : {len( corrupted_train )}"
)

print(
	f"Corrupted test images  : {len( corrupted_test )}"
)

print(
	f"DataLoader batch shape : {tuple( images.shape )}"
)

print( "=" * 60 )

# Overall sanity check

if (

	len( missing_train ) == 0

	and len( missing_test ) == 0

	and len( corrupted_train ) == 0

	and len( corrupted_test ) == 0

	and train_df["label"].nunique() == 200

	and test_df["label"].nunique() == 200

):

	print(
		"Dataset sanity check: PASSED"
	)

else:

	print(
		"Dataset sanity check: "
		"PLEASE CHECK THE WARNINGS ABOVE"
	)

In [ ]:
print( "Training images per class:" )
print( train_counts.describe() )

print( "\nTesting images per class:" )
print( test_counts.describe() )

print( "Training class counts:" )
display( train_counts.to_frame( "count" ) )

# Conclusion

The CUB-200 dataset has been inspected before model training.

The following aspects were checked:

1. Dataset directory structure
2. Training and testing annotation files
3. Number of images
4. Number of classes
5. Class distribution
6. Image availability
7. Image dimensions
8. Corrupted images
9. PyTorch Dataset
10. PyTorch DataLoader
11. Data augmentation

If all checks pass, the dataset is ready for the model training stage.

The actual training process will be performed using:

```text
src/train.py
```

The model architecture will be defined in:

```text
src/models.py
```